# Data Quality Assessment

## Purpose

The purpose of this notebook is to evaluate the quality, completeness, and reliability of the master Fitbit dataset before conducting exploratory analysis.

This assessment will focus on identifying missing values, validating data types, evaluating participant coverage, detecting potential outliers, and documenting any limitations that could influence the interpretation of results.

The findings from this notebook will guide data cleaning decisions and establish confidence in the dataset used throughout the remainder of the project.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
master_daily_activity = pd.read_csv(
    r"C:\Users\Weekseey\Documents\Data Projects\bellabeat-analysis\data\processed\master_daily_activity.csv"
)

master_daily_activity["ActivityDate"] = pd.to_datetime(
    master_daily_activity["ActivityDate"]
)

## Dataset Overview

Before evaluating data quality issues, I will review the structure and dimensions of the consolidated master dataset.

In [3]:
print("Rows:", master_daily_activity.shape[0])
print("Columns:", master_daily_activity.shape[1])

print("\nUnique Users:")
print(master_daily_activity["Id"].nunique())

print("\nDate Range:")
print(master_daily_activity["ActivityDate"].min())
print(master_daily_activity["ActivityDate"].max())

Rows: 1373
Columns: 15

Unique Users:
35

Date Range:
2016-03-12 00:00:00
2016-05-12 00:00:00


## Missing Value Assessment

Missing values can reduce analytical reliability and may require cleaning or special handling during analysis.

The first step is to identify whether any fields contain missing observations.

In [4]:
missing_values = master_daily_activity.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Percent Missing": round(
        missing_values / len(master_daily_activity) * 100,
        2
    )
})

missing_summary

,Missing Values,Percent Missing
Id,0,0.0
ActivityDate,0,0.0
TotalSteps,0,0.0
TotalDistance,0,0.0
TrackerDistance,0,0.0
LoggedActivitiesDistance,0,0.0
VeryActiveDistance,0,0.0
ModeratelyActiveDistance,0,0.0
LightActiveDistance,0,0.0
SedentaryActiveDistance,0,0.0


## Missing Value Findings

No missing values were identified in the master daily activity dataset.

All 15 variables contained complete records across the 1,373 observations.

While no null values were present, several activity-related variables contain zero values. These observations may represent legitimate inactivity, device non-use, or incomplete tracking behavior and will be investigated further during the data quality assessment process.

## Zero Activity Assessment

Because wearable device datasets often use zero values instead of missing values, it is important to evaluate how frequently users recorded no activity.

High frequencies of zero activity may indicate device non-use rather than true inactivity.

In [5]:
zero_steps = (master_daily_activity["TotalSteps"] == 0).sum()

print("Days with 0 steps:", zero_steps)

print(
    "Percent of records:",
    round(
        zero_steps / len(master_daily_activity) * 100,
        2
    ),
    "%"
)

Days with 0 steps: 133
Percent of records: 9.69 %


In [6]:
master_daily_activity["TotalSteps"].describe()

count     1373.000000
mean      7377.375091
std       5198.133424
min          0.000000
25%       3321.000000
50%       7142.000000
75%      10645.000000
max      36019.000000
Name: TotalSteps, dtype: float64

## Zero Activity Findings

The master dataset contains 133 records with zero recorded steps, representing approximately 9.7% of all observations.

Although zero-step days may indicate device non-use or incomplete tracking, the proportion is relatively small compared to the overall dataset. At this stage, there is insufficient evidence to classify these observations as invalid records.

Zero-step days will be retained for analysis and investigated further alongside other activity metrics.

In [7]:
zero_step_users = (
    master_daily_activity[master_daily_activity["TotalSteps"] == 0]
    .groupby("Id")
    .size()
    .sort_values(ascending=False)
)

zero_step_users

Id
1927972279    14
4020332650    14
4057192912    14
6290855005    13
1844505072    12
8792009665    10
6775888955    10
4388161847     8
6117666160     7
2891001357     6
8253242879     6
6391747486     6
2320127002     4
7007744171     2
5577150313     2
4702921684     1
7086361926     1
4319703577     1
8583815059     1
1503960366     1
dtype: int64

## Zero-Step User Distribution

Although only 9.7% of all records contained zero recorded steps, these observations were not evenly distributed across participants.

Several users recorded 10 or more zero-step days during the study period, suggesting that inactivity or device non-use may be concentrated among a small subset of participants rather than occurring uniformly across the dataset.

At this stage, these records will be retained. Additional investigation is required to determine whether zero-step days represent legitimate inactivity or periods when the Fitbit device was not actively used.

In [8]:
user_days = (
    master_daily_activity.groupby("Id")
    .size()
    .sort_values()
)

user_days

Id
2891001357     8
6391747486     9
3372868164    30
8253242879    30
2347167796    32
4057192912    35
6775888955    35
7007744171    37
6117666160    38
8583815059    39
6290855005    39
4388161847    39
1644430081    40
8792009665    40
5577150313    41
8053475328    41
3977333714    41
7086361926    42
8378563200    42
4558609924    42
5553957443    42
2873212765    42
2320127002    42
2026352035    42
2022484408    42
1927972279    42
1844505072    42
8877689391    42
4319703577    43
6962181067    44
4445114986    45
4702921684    45
1624580081    49
1503960366    49
4020332650    62
dtype: int64

In [9]:
user_days.describe()

count    35.000000
mean     39.228571
std       9.490198
min       8.000000
25%      38.500000
50%      42.000000
75%      42.000000
max      62.000000
dtype: float64

## Participant Coverage Assessment

Participant contribution levels varied considerably across the dataset.

While several users contributed activity records throughout the full 62-day observation period, others contributed fewer than 10 days of data.

### Key Findings

- Minimum participant coverage: 8 days
- Maximum participant coverage: 62 days
- Median participant coverage: 42 days
- Average participant coverage: 39.2 days

The variation in participation suggests that some users may have discontinued device usage, stopped syncing activity data, or participated for only a portion of the study period.

These users will be retained during the assessment phase, but participant coverage should be considered when interpreting activity trends and summary statistics.

In [10]:
user_summary = pd.DataFrame({
    "DaysRecorded": master_daily_activity.groupby("Id").size(),
    "ZeroStepDays": master_daily_activity.groupby("Id")["TotalSteps"]
        .apply(lambda x: (x == 0).sum())
})

user_summary["PercentZeroDays"] = round(
    user_summary["ZeroStepDays"] /
    user_summary["DaysRecorded"] * 100,
    2
)

user_summary.sort_values(
    "PercentZeroDays",
    ascending=False
).head(15)

,DaysRecorded,ZeroStepDays,PercentZeroDays
Id,,,
2891001357,8,6,75.00
6391747486,9,6,66.67
4057192912,35,14,40.00
1927972279,42,14,33.33
6290855005,39,13,33.33
1844505072,42,12,28.57
6775888955,35,10,28.57
8792009665,40,10,25.00
4020332650,62,14,22.58


## Participant Activity Quality Findings

An analysis of participant activity patterns revealed that zero-step days were concentrated among a relatively small subset of users.

Two participants recorded fewer than 10 days of activity data while also exhibiting extremely high proportions of zero-step days (greater than 65%). Several additional users recorded zero-step days on more than 25% of their observed dates.

These findings may indicate periods of device non-use, incomplete synchronization, or reduced participant engagement. However, because these observations may represent genuine user behavior, the records will be retained for analysis.

The presence of low-engagement users should be considered when interpreting activity trends and behavioral patterns.

In [11]:
master_daily_activity[
    [
        "TotalSteps",
        "Calories",
        "SedentaryMinutes",
        "VeryActiveMinutes"
    ]
].describe()

,TotalSteps,Calories,SedentaryMinutes,VeryActiveMinutes
count,1373.000000,1373.000000,1373.000000,1373.000000
mean,7377.375091,2294.812090,1001.346686,19.871085
std,5198.133424,725.526989,304.168197,31.817631
min,0.000000,0.000000,0.000000,0.000000
25%,3321.000000,1820.000000,734.000000,0.000000
50%,7142.000000,2129.000000,1062.000000,2.000000
75%,10645.000000,2781.000000,1246.000000,30.000000
max,36019.000000,4900.000000,1440.000000,210.000000


## Data Quality Assessment Summary

The master daily activity dataset was evaluated for completeness, consistency, participant coverage, and potential data quality issues.

### Key Findings

- No missing values were identified across any variables.
- No duplicate user-date records remained after dataset consolidation.
- Activity metrics fell within reasonable ranges and no impossible values were detected.
- Participant coverage varied substantially, ranging from 8 to 62 recorded days.
- Zero-step days represented approximately 9.7% of observations and were concentrated among a small subset of participants.
- Some observations may reflect periods of device non-use rather than true inactivity.

### Conclusion

The dataset is suitable for exploratory analysis and trend identification. While participant engagement varies across users, no critical data quality issues were identified that would prevent meaningful analysis.

All records will be retained for analysis, with documented limitations considered when interpreting results.